In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "customer360_dev.bronze.support_tickets"
)

silver_df = spark.table(
    "customer360_dev.silver.support_tickets"
)

reject_df = spark.table(
    "customer360_dev.quarantine.support_ticket_rejects"
)

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
reject_count = reject_df.count()

print(f"Bronze rows   : {bronze_count:,}")
print(f"Silver rows   : {silver_count:,}")
print(f"Rejected rows : {reject_count:,}")

Bronze rows   : 1,500,096
Silver rows   : 952,329
Rejected rows : 547,767


In [0]:
customers_df = (
    spark.table("customer360_dev.silver.customers")
    .select("customer_id")
)

orders_df = (
    spark.table("customer360_dev.silver.orders")
    .select("order_id")
)

validated_df = (
    bronze_df
    .withColumn(
        "created_timestamp_cast",
        F.to_timestamp("created_timestamp")
    )
    .withColumn(
        "first_response_timestamp_cast",
        F.to_timestamp("first_response_timestamp")
    )
    .withColumn(
        "resolved_timestamp_cast",
        F.to_timestamp("resolved_timestamp")
    )
    .withColumn(
        "closed_timestamp_cast",
        F.to_timestamp("closed_timestamp")
    )
    .withColumn(
        "csat_score_cast",
        F.col("customer_satisfaction_score").cast("int")
    )
    .withColumn(
        "priority_clean",
        F.upper(F.trim("ticket_priority"))
    )
    .withColumn(
        "ticket_status_clean",
        F.upper(F.trim("ticket_status"))
    )
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
    .join(
        orders_df.withColumn(
            "_order_exists",
            F.lit(True)
        ),
        on="order_id",
        how="left"
    )
)

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_ticket_id",
        F.col("ticket_id").isNull()
        | (F.trim(F.col("ticket_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_order_not_found",
        F.col("order_id").isNotNull()
        & F.col("_order_exists").isNull()
    )
    .withColumn(
        "dq_invalid_priority",
        ~F.col("priority_clean").isin(
            "LOW",
            "MEDIUM",
            "HIGH",
            "CRITICAL"
        )
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("ticket_status_clean").isin(
            "OPEN",
            "IN_PROGRESS",
            "RESOLVED",
            "CLOSED",
            "REOPENED"
        )
    )
    .withColumn(
        "dq_invalid_csat",
        F.col("csat_score_cast").isNotNull()
        & (
            (F.col("csat_score_cast") < 1)
            | (F.col("csat_score_cast") > 5)
        )
    )
)

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_response_before_created",
        F.col("first_response_timestamp_cast").isNotNull()
        & (
            F.col("first_response_timestamp_cast")
            < F.col("created_timestamp_cast")
        )
    )
    .withColumn(
        "dq_resolved_before_created",
        F.col("resolved_timestamp_cast").isNotNull()
        & (
            F.col("resolved_timestamp_cast")
            < F.col("created_timestamp_cast")
        )
    )
    .withColumn(
        "dq_closed_before_resolved",
        F.col("closed_timestamp_cast").isNotNull()
        & F.col("resolved_timestamp_cast").isNotNull()
        & (
            F.col("closed_timestamp_cast")
            < F.col("resolved_timestamp_cast")
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_ticket_id")
    | F.col("dq_missing_customer_id")
    | F.col("dq_customer_not_found")
    | F.col("dq_order_not_found")
    | F.col("dq_invalid_priority")
    | F.col("dq_invalid_status")
    | F.col("dq_invalid_csat")
    | F.col("dq_response_before_created")
    | F.col("dq_resolved_before_created")
    | F.col("dq_closed_before_resolved")
)

In [0]:
valid_candidate_count = (
    validated_df
    .filter(~F.col("dq_critical_failure"))
    .count()
)

duplicate_rows_removed = (
    valid_candidate_count - silver_count
)

print(
    f"Valid candidates before dedup: "
    f"{valid_candidate_count:,}"
)

print(
    f"Duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Valid candidates before dedup: 952,329
Duplicate rows removed: 0


In [0]:
accounted_for = (
    silver_count
    + reject_count
    + duplicate_rows_removed
)

difference = bronze_count - accounted_for

print(f"Bronze rows        : {bronze_count:,}")
print(f"Silver rows        : {silver_count:,}")
print(f"Rejected rows      : {reject_count:,}")
print(f"Duplicates removed : {duplicate_rows_removed:,}")
print(f"Accounted for      : {accounted_for:,}")
print(f"Difference         : {difference:,}")

Bronze rows        : 1,500,096
Silver rows        : 952,329
Rejected rows      : 547,767
Duplicates removed : 0
Accounted for      : 1,500,096
Difference         : 0


In [0]:
result = [
    {
        "entity": "support_tickets",
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "quarantine_rows": reject_count,
        "duplicates_removed": duplicate_rows_removed,
        "accounted_for_rows": accounted_for,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    }
]

display(
    spark.createDataFrame(result)
)

accounted_for_rows,bronze_rows,difference,duplicates_removed,entity,quarantine_rows,silver_rows,status
1500096,1500096,0,0,support_tickets,547767,952329,PASS


In [0]:
orphan_customers = (
    silver_df
    .join(
        spark.table(
            "customer360_dev.silver.customers"
        ).select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Support Tickets with missing Silver Customer:",
    orphan_customers
)

Support Tickets with missing Silver Customer: 0


In [0]:
orphan_orders = (
    silver_df
    .filter(F.col("order_id").isNotNull())
    .join(
        spark.table(
            "customer360_dev.silver.orders"
        ).select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .count()
)

print(
    "Support Tickets with invalid non-null Order:",
    orphan_orders
)

Support Tickets with invalid non-null Order: 0


In [0]:
warning_summary = (
    silver_df
    .select(
        F.sum(
            F.when(
                F.col(
                    "dq_warning_resolved_without_timestamp"
                ),
                1
            ).otherwise(0)
        ).alias(
            "resolved_without_timestamp"
        ),

        F.sum(
            F.when(
                F.col(
                    "dq_warning_closed_without_timestamp"
                ),
                1
            ).otherwise(0)
        ).alias(
            "closed_without_timestamp"
        )
    )
)

display(warning_summary)

resolved_without_timestamp,closed_without_timestamp
0,0


In [0]:
(
    reject_df
    .groupBy("rejection_reason")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+-----------------------------------------------------------------------------------+------+
|rejection_reason                                                                   |count |
+-----------------------------------------------------------------------------------+------+
|RESOLVED_BEFORE_CREATED                                                            |241212|
|CLOSED_BEFORE_RESOLVED                                                             |181959|
|RESPONSE_BEFORE_CREATED; RESOLVED_BEFORE_CREATED                                   |57000 |
|ORDER_NOT_FOUND                                                                    |24908 |
|CUSTOMER_NOT_FOUND                                                                 |14964 |
|ORDER_NOT_FOUND; RESOLVED_BEFORE_CREATED                                           |6592  |
|ORDER_NOT_FOUND; RESPONSE_BEFORE_CREATED; RESOLVED_BEFORE_CREATED                  |4500  |
|CUSTOMER_NOT_FOUND; RESOLVED_BEFORE_CREATED                          